# Category performance

Category sizes, co-occurrence, and subcategory recall in the held-out sentence predictions.

In [ ]:
from shared import paths
from IPython.display import display


## Setup

In [ ]:
paths.prepare_directories()


## Category size and decision thresholds

In [ ]:
INPUT_CSV = str(paths.sentence_run(4) / 'general/all_folds_predictions.csv')
OUTPUT_PATH = str(paths.FIGURES_DIR / 'category_frequency_and_f1.png')
import ast
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_fscore_support
import matplotlib.lines as mlines

df = pd.read_csv(INPUT_CSV)
df['true_labels'] = df['true_labels'].apply(ast.literal_eval)
df['probabilities'] = df['probabilities'].apply(ast.literal_eval)
y_true = np.array(df['true_labels'].tolist(), dtype=int)
y_prob = np.array(df['probabilities'].tolist(), dtype=float)
CATEGORIES_ORIG = [
    'Editorial and Publishing Processes',
    'Elements of Fictional Craft',
    'Financial and Legal Aspects',
    'Gratitude and Acknowledgment',
    'Impact and Reception',
    'Inspirations and Influences',
    'Marketing and Promotion',
    'Quality and Evaluation',
    'Writing Process and Creative Insight',
]
if y_true.shape[1] != len(CATEGORIES_ORIG):
    CATEGORIES_ORIG = [f'Category {i + 1}' for i in range(y_true.shape[1])]


def binarise(probs, thr):
    return (probs >= thr).astype(int)


def per_category_prf(y_true_bin, y_pred_bin):
    p, r, f1, support = precision_recall_fscore_support(y_true_bin, y_pred_bin, average=None, zero_division=0)
    return (p, r, f1, support)
instances = y_true.sum(axis=0).astype(int)
C = instances.shape[0]
thresholds = np.array([round(i / 100, 2) for i in range(5, 100, 5)])
f1_best = np.zeros(C, dtype=float)
best_tau = np.zeros(C, dtype=float)
for j in range(C):
    f1s_j = []
    for tau in thresholds:
        y_pred_tau = binarise(y_prob, tau)
        _, _, f1_vec, _ = per_category_prf(y_true, y_pred_tau)
        f1s_j.append(f1_vec[j])
    f1s_j = np.array(f1s_j)
    k = int(f1s_j.argmax())
    f1_best[j] = f1s_j[k]
    best_tau[j] = float(thresholds[k])
df_corr = pd.DataFrame(
    {
        'category': CATEGORIES_ORIG,
        'instances': instances,
        'f1_at_best': f1_best,
        'best_tau': best_tau,
    },
)
ORDER_NEW = [
    'Inspirations and Influences',
    'Writing Process and Creative Insight',
    'Elements of Fictional Craft',
    'Editorial and Publishing Processes',
    'Quality and Evaluation',
    'Financial and Legal Aspects',
    'Gratitude and Acknowledgment',
    'Marketing and Promotion',
    'Impact and Reception',
]
df_corr = df_corr.set_index('category').loc[ORDER_NEW].reset_index()
df_corr.insert(0, 'idx', np.arange(1, len(df_corr) + 1))
pearson_best = df_corr[['instances', 'f1_at_best']].corr(method='pearson').iloc[0, 1]
spearman_best = df_corr[['instances', 'f1_at_best']].corr(method='spearman').iloc[0, 1]
print('\n=== Correlation (instances vs. F1 @ per-category best tau) ===')
print(f'Pearson:  {pearson_best:.3f}')
print(f'Spearman: {spearman_best:.3f}')
print('\nPer-category summary:')
print(df_corr.to_string(index=False, float_format='%.4f'))
sns.set_theme(style='darkgrid')
plt.figure(figsize=(8.5, 6.5))
sns.regplot(
    x='instances',
    y='f1_at_best',
    data=df_corr,
    ci=None,
    scatter_kws={'s': 60, 'color': 'black'},
    line_kws={'color': 'black'},
)
for _, row in df_corr.iterrows():
    plt.annotate(
        str(int(row['idx'])),
        (row['instances'], row['f1_at_best']),
        xytext=(4, 5),
        textcoords='offset points',
        fontsize=9,
        color='black',
    )
plt.xlabel('Instances')
plt.ylabel('F1')
legend_elems = [
    mlines.Line2D([], [], color='none', label=f'{int(i)}: {cat}')
     for i, cat in zip(df_corr['idx'], df_corr['category'])
]
plt.legend(
    handles=legend_elems,
    loc='upper center',
    bbox_to_anchor=(0.5, -0.14),
    frameon=True,
    prop={'size': 9},
    ncol=3,
)
plt.tight_layout()
plt.subplots_adjust(bottom=0.28)
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
plt.savefig(OUTPUT_PATH, dpi=300, bbox_inches='tight')
plt.show()


## Category co-occurrence

In [ ]:
INPUT_CSV = str(paths.sentence_run(4) / 'general/all_folds_predictions.csv')
CATEGORIES_DEFAULT = [
    'Editorial and Publishing Processes',
    'Elements of Fictional Craft',
    'Financial and Legal Aspects',
    'Gratitude and Acknowledgment',
    'Impact and Reception',
    'Inspirations and Influences',
    'Marketing and Promotion',
    'Quality and Evaluation',
    'Writing Process and Creative Insight',
]
CATEGORY_THRESHOLDS = best_tau.copy()
import ast
import numpy as np
import pandas as pd
from typing import Tuple
from sklearn.metrics import precision_recall_fscore_support


def parse_list_column(series: pd.Series) -> np.ndarray:
    parsed = series.apply(lambda s: ast.literal_eval(s) if isinstance(s, str) else s)
    return np.array(parsed.to_list())


def get_columns(df: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray]:
    cand_true = ['true_labels', 'y_true', 'labels_true', 'labels']
    cand_prob = ['probabilities', 'y_prob', 'probs', 'pred_proba']
    col_true = next((c for c in cand_true if c in df.columns), None)
    col_prob = next((c for c in cand_prob if c in df.columns), None)
    if col_true is None or col_prob is None:
        raise ValueError(
            f"Missing required columns. Have: {list(df.columns)}. Need e.g. 'true_labels' and 'probabilities'.",
        )
    y_true = parse_list_column(df[col_true]).astype(int)
    y_prob = parse_list_column(df[col_prob]).astype(float)
    if y_true.shape != y_prob.shape:
        raise ValueError(f'Shape mismatch: y_true {y_true.shape} vs y_prob {y_prob.shape}')
    return (y_true, y_prob)


def infer_categories(n_classes: int) -> list:
    return CATEGORIES_DEFAULT if n_classes == len(CATEGORIES_DEFAULT) else [f'Category {i}' for i in range(n_classes)]


def binarise_per_class(y_prob: np.ndarray, taus: np.ndarray) -> np.ndarray:
    return (y_prob >= taus[None, :]).astype(int)


def subset_isolated(y_true: np.ndarray) -> np.ndarray:
    return y_true.sum(axis=1) == 1


def subset_with_partner(y_true: np.ndarray, j: int) -> np.ndarray:
    return y_true[:, j] == 1


def prf_binary(y_true_bin: np.ndarray, y_pred_bin: np.ndarray):
    p, r, f, _ = precision_recall_fscore_support(y_true_bin, y_pred_bin, average='binary', zero_division=0)
    return (float(p), float(r), float(f))


def main():
    df = pd.read_csv(INPUT_CSV)
    y_true, y_prob = get_columns(df)
    N, C = y_true.shape
    categories = infer_categories(C)
    if C != len(CATEGORY_THRESHOLDS):
        raise ValueError(f'Mismatch: {C} categories but {len(CATEGORY_THRESHOLDS)} thresholds defined.')
    taus = np.array(CATEGORY_THRESHOLDS, dtype=float)
    y_pred = binarise_per_class(y_prob, taus)
    tau_df = pd.DataFrame({'category': categories, 'threshold': taus})
    print('\n=== Selected category thresholds ===')
    print(tau_df.to_string(index=False, float_format='%.2f'))
    rows = []
    mask_iso_global = subset_isolated(y_true)
    for k, cat in enumerate(categories):
        y_t_all = y_true[:, k]
        y_p_all = y_pred[:, k]
        supp_true_all = int(y_t_all.sum())
        p_all, r_all, f_all = prf_binary(y_t_all, y_p_all)
        rows.append(
            {
                'base_category': cat,
                'partner': 'overall',
                'support_true': supp_true_all,
                'precision': p_all,
                'recall': r_all,
                'f1': f_all,
            },
        )
        y_t_iso = y_true[mask_iso_global, k]
        y_p_iso = y_pred[mask_iso_global, k]
        supp_true_iso = int(y_t_iso.sum())
        p, r, f = prf_binary(y_t_iso, y_p_iso)
        rows.append(
            {
                'base_category': cat,
                'partner': 'isolated',
                'support_true': supp_true_iso,
                'precision': p,
                'recall': r,
                'f1': f,
            },
        )
        for j, partner in enumerate(categories):
            if j == k:
                continue
            mask_ctx = subset_with_partner(y_true, j)
            y_t = y_true[mask_ctx, k]
            y_p = y_pred[mask_ctx, k]
            supp_true = int(y_t.sum())
            p, r, f = prf_binary(y_t, y_p)
            rows.append(
                {
                    'base_category': cat,
                    'partner': partner,
                    'support_true': supp_true,
                    'precision': p,
                    'recall': r,
                    'f1': f,
                },
            )
    out = pd.DataFrame(rows)

    def partner_rank(x):
        if x == 'overall':
            return 0
        elif x == 'isolated':
            return 1
        else:
            return 2
    out['partner_rank'] = out['partner'].map(partner_rank)
    out = out.sort_values(
        ['base_category', 'partner_rank', 'support_true'],
        ascending=[True, True, False],
    ).drop(
        columns=['partner_rank'],
    )
    pd.set_option('display.max_rows', None)
    print('\n=== Context-conditioned performance for each base category (detecting c) ===')
    print(out.to_string(index=False, float_format='%.4f'))
if __name__ == '__main__':
    main()


## Attach subcategories

In [ ]:
import pandas as pd
import re
import ast
import json
from collections import defaultdict
from shared.utils import load_pickle
from shared.labels import CATEGORY_INDEX

NUM_CATEGORIES = len(CATEGORY_INDEX)
PREDICTIONS_CSV = str(paths.sentence_run(4) / 'general/all_folds_predictions.csv')
OUTPUT_CSV = str(paths.sentence_run(4) / 'general/all_folds_predictions_with_subcategories.csv')
OUTPUT_MAPPING_JSON = str(paths.sentence_run(4) / 'general/subcategory_to_maincategory.json')
DIVIDER = '_' * 60
ABBREVIATIONS = [
    'Mr',
    'Mrs',
    'Ms',
    'Dr',
    'Prof',
    'Sr',
    'Jr',
    'St',
    'vs',
    'i.e',
    'e.g',
    'No',
    'Dept',
    'Inc',
    'Ltd',
    'Jan',
    'Feb',
    'Mar',
    'Apr',
    'Jun',
    'Jul',
    'Aug',
    'Sep',
    'Sept',
    'Oct',
    'Nov',
    'Dec',
    'Vol',
]
from shared.text import protect_abbreviations
from shared.text import restore_abbreviations


def extract_subcategories():
    letters = load_pickle()
    reviewed_letters = [l for l in letters if getattr(l, 'reviewed', False) and l.intertextual_traces]
    result = dict()
    subcat_to_maincat = dict()
    for letter in reviewed_letters:
        anno_text = letter.anno_text
        match = re.search(f'{DIVIDER}\\n\\n(.*?)\\n\\n{DIVIDER}', anno_text, re.DOTALL)
        if not match:
            continue
        rel_text = match.group(1)
        rel_start = match.start(1)
        rel_text = protect_abbreviations(rel_text, ABBREVIATIONS)
        sentence_pattern = re.compile('.{6,}?[.!?;](?:["\\\'*]*)?(?:\\[\\[\\d+\\]\\])?(?=\\s|$)', re.DOTALL)
        segments = sentence_pattern.findall(rel_text)
        remainder = rel_text[sum((len(s) for s in segments)):].strip()
        if remainder:
            segments.append(remainder)
        sentences = [restore_abbreviations(s) for s in segments]
        sentences = [re.sub('\\[\\[\\d+\\]\\]$', '', s).strip() for s in sentences if s.strip()]
        offset = rel_start
        sentence_data = []
        for sent in sentences:
            sent_start = anno_text.find(sent, offset)
            if sent_start == -1:
                continue
            sent_end = sent_start + len(sent)
            offset = sent_end
            sentence_data.append((sent_start, sent_end, sent))
        for i, (sent_start, sent_end, _) in enumerate(sentence_data):
            subcats = set()
            for trace in letter.intertextual_traces:
                for ann in trace.annotations:
                    if not ann.category:
                        continue
                    if ann.end_offset <= sent_start or ann.start_offset >= sent_end:
                        continue
                    if ann.subcategory:

                        def register(sub):
                            if isinstance(sub, str) and sub not in subcat_to_maincat:
                                subcat_to_maincat[sub] = ann.category
                        if isinstance(ann.subcategory, list):
                            subcats.update([sc for sc in ann.subcategory if isinstance(sc, str)])
                            for sc in ann.subcategory:
                                register(sc)
                        elif isinstance(ann.subcategory, str):
                            subcats.add(ann.subcategory)
                            register(ann.subcategory)
            result[letter.id, i] = sorted(subcats)
    return (result, subcat_to_maincat)


def main():
    print(' Loading predictions...')
    df = pd.read_csv(PREDICTIONS_CSV)
    if 'true_labels' in df:
        df['true_labels'] = df['true_labels'].apply(ast.literal_eval)
    if 'probabilities' in df:
        df['probabilities'] = df['probabilities'].apply(ast.literal_eval)
    print(' Extracting subcategories from annotations...')
    subcat_map, subcat_to_maincat = extract_subcategories()
    print(' Attaching subcategories...')

    def get_subcats(row):
        return subcat_map.get((row['letter_id'], row['sentence_id']), [])
    df['subcategories'] = df.apply(get_subcats, axis=1)
    print(f' Saving CSV to:\n{OUTPUT_CSV}')
    df.to_csv(OUTPUT_CSV, index=False)
    print(f' Saving the subcategory-to-category mapping to:\n{OUTPUT_MAPPING_JSON}')
    with open(OUTPUT_MAPPING_JSON, 'w') as f:
        json.dump(subcat_to_maincat, f, indent=2)
    print(' Done.')
if __name__ == '__main__':
    main()


## Subcategory recall

In [ ]:
import pandas as pd
import json
import ast
from collections import defaultdict

CSV_PATH = str(paths.sentence_run(4) / 'general/all_folds_predictions_with_subcategories.csv')
JSON_PATH = str(paths.sentence_run(4) / 'general/subcategory_to_maincategory.json')
from shared.labels import CATEGORY_INDEX

BEST_TAU = {name: float(best_tau[i]) for name, i in CATEGORY_INDEX.items()}
print(' Loading data...')
df = pd.read_csv(CSV_PATH)
df['true_labels'] = df['true_labels'].apply(ast.literal_eval)
df['probabilities'] = df['probabilities'].apply(ast.literal_eval)
df['subcategories'] = df['subcategories'].apply(ast.literal_eval)
with open(JSON_PATH, 'r') as f:
    sub_to_main = json.load(f)
grouped_results = defaultdict(list)
for subcat in sorted(sub_to_main.keys()):
    maincat = sub_to_main[subcat]
    idx = CATEGORY_INDEX[maincat]
    tau = BEST_TAU[maincat]
    df_sub = df[df['subcategories'].apply(lambda lst: subcat in lst)]
    TP = 0
    FN = 0
    for _, row in df_sub.iterrows():
        y_true = row['true_labels'][idx]
        y_prob = row['probabilities'][idx]
        y_pred = 1 if y_prob >= tau else 0
        if y_true == 1 and y_pred == 1:
            TP += 1
        elif y_true == 1 and y_pred == 0:
            FN += 1
    total = TP + FN
    recall = TP / total if total > 0 else 0.0
    grouped_results[maincat].append(
        {'subcategory': subcat, 'count': total, 'TP': TP, 'FN': FN, 'recall': round(recall, 3)},
    )
print('\n Recall by category and subcategory at the selected thresholds:\n')
for main_cat in CATEGORY_INDEX.keys():
    print(f'▶  {main_cat}')
    sub_entries = grouped_results.get(main_cat, [])
    if not sub_entries:
        print('   (no assigned subcategories)')
        continue
    total_TP = sum((e['TP'] for e in sub_entries))
    total_FN = sum((e['FN'] for e in sub_entries))
    total_count = total_TP + total_FN
    total_recall = total_TP / total_count if total_count > 0 else 0.0
    for entry in sorted(sub_entries, key=lambda x: -x['recall']):
        print(
            f"   - {entry['subcategory']:30s}  (n={entry['count']:4d})   TP={entry['TP']:3d}  FN={entry['FN']:3d}  Recall={entry['recall']:.3f}",
        )
    print(f"   {'-' * 70}")
    print(
        f"   {'Total':30s}  (n={total_count:4d})   TP={total_TP:3d}  FN={total_FN:3d}  Recall={total_recall:.3f}",
    )
    print()
